In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import os
import torch.nn.functional as F
from torch import optim
from collections import defaultdict
import time
import datetime
import sys
# Device setup
torch._C._jit_set_bailout_depth(2)

if torch.backends.mps.is_available():
    print("MPS backend is available!")
    device = torch.device("mps")
else:
    print("MPS backend is not available. Using CUDA if available, otherwise CPU.")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
import json
import matplotlib.pyplot as plt



class SimpleTemporalEncoding(nn.Module):
    """Simplified temporal encoding module"""
    def __init__(self, d_model, max_len=100):
        super().__init__()
        # Create position embeddings to learn temporal patterns
        self.position_embeddings = nn.Parameter(torch.randn(max_len, d_model))
        
    def forward(self, x):
        # x shape: [batch_size, seq_len, d_model]
        seq_len = x.size(1)
        # Add positional embeddings
        return x + self.position_embeddings[:seq_len]

class EnhancedTransformerSequentialModule(nn.Module):
    """Enhanced sequential learning module with dynamic transformer encoder"""
    def __init__(self, input_dim, hidden_dim, nhead=4, num_layers=2, dropout=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        
        # Project input to hidden dimension if needed
        self.input_projection = nn.Linear(input_dim, hidden_dim) if input_dim != hidden_dim else nn.Identity()
        
        # Stack of transformer encoder blocks
        self.layers = nn.ModuleList()
        for _ in range(num_layers):
            self.layers.append(nn.ModuleDict({
                'attention': DynamicMultiHeadAttention(hidden_dim, nhead),
                'norm1': nn.LayerNorm(hidden_dim),
                'feedforward': nn.Sequential(
                    nn.Linear(hidden_dim, hidden_dim * 4),
                    nn.GELU(),
                    nn.Dropout(dropout),
                    nn.Linear(hidden_dim * 4, hidden_dim),
                    nn.Dropout(dropout)
                ),
                'norm2': nn.LayerNorm(hidden_dim)
            }))
        
        # Attention for sequence aggregation
        self.attention = nn.Linear(hidden_dim, 1)
    
    def forward(self, x):
        # Project input if dimensions don't match
        x = self.input_projection(x)
        
        # Process with transformer encoder blocks
        for layer in self.layers:
            # Self-attention with residual connection
            attn_output, _ = layer['attention'](x)
            x = layer['norm1'](x + attn_output)
            
            # Feed-forward with residual connection
            ff_output = layer['feedforward'](x)
            x = layer['norm2'](x + ff_output)
        
        # Apply attention for sequence aggregation
        attn_weights = F.softmax(self.attention(x), dim=1)
        context = torch.sum(x * attn_weights, dim=1)
        
        return context, x

class DynamicMultiHeadAttention(nn.Module):
    """Enhanced multi-head attention with dynamic graph structure"""
    def __init__(self, embed_dim, num_heads=4):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        # Make sure embed_dim is divisible by num_heads
        assert embed_dim % num_heads == 0, "Embedding dimension must be divisible by number of heads"
        
        # Linear projections for queries, keys, values
        self.query_projection = nn.Linear(embed_dim, embed_dim)
        self.key_projection = nn.Linear(embed_dim, embed_dim)
        self.value_projection = nn.Linear(embed_dim, embed_dim)
        
        # Output projection
        self.output_projection = nn.Linear(embed_dim, embed_dim)
        
        # Scaling factor for dot-product attention
        self.scaling = self.head_dim ** -0.5
    
    def forward(self, x, mask=None):
        """
        x: Input of shape (batch_size, seq_len, embed_dim)
        mask: Optional mask to apply to attention scores
        """
        batch_size, seq_len, _ = x.size()
        
        # Project queries, keys, values
        # Shape: (batch_size, seq_len, num_heads, head_dim)
        queries = self.query_projection(x).view(batch_size, seq_len, self.num_heads, self.head_dim)
        keys = self.key_projection(x).view(batch_size, seq_len, self.num_heads, self.head_dim)
        values = self.value_projection(x).view(batch_size, seq_len, self.num_heads, self.head_dim)
        
        # Transpose for attention calculation
        # Shape: (batch_size, num_heads, seq_len, head_dim)
        queries = queries.transpose(1, 2)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)
        
        # Calculate attention scores
        # scores: (batch_size, num_heads, seq_len, seq_len)
        scores = torch.matmul(queries, keys.transpose(-2, -1)) * self.scaling
        
        # Apply mask if provided
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
            
        # Apply softmax to get attention weights
        attn_weights = F.softmax(scores, dim=-1)
        
        # Apply attention weights to values
        # context: (batch_size, num_heads, seq_len, head_dim)
        context = torch.matmul(attn_weights, values)
        
        # Transpose and reshape
        # output: (batch_size, seq_len, embed_dim)
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.embed_dim)
        output = self.output_projection(context)
        
        return output, attn_weights

class EnhancedSectorAttention(nn.Module):
    """Enhanced attention mechanism for sector modeling with graph awareness"""
    def __init__(self, embed_dim, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.embed_dim = embed_dim
        self.head_dim = embed_dim // num_heads
        
        # Multi-head attention components
        self.query_proj = nn.Linear(embed_dim, embed_dim)
        self.key_proj = nn.Linear(embed_dim, embed_dim)
        self.value_proj = nn.Linear(embed_dim, embed_dim)
        self.output_proj = nn.Linear(embed_dim, embed_dim)
        
        self.scaling = self.head_dim ** -0.5
        
    def forward(self, x, mask=None):
        # x shape: [batch_size, num_nodes, embed_dim]
        batch_size, num_nodes, _ = x.size()
        
        # Compute Q, K, V with multi-head separation
        q = self.query_proj(x).view(batch_size, num_nodes, self.num_heads, self.head_dim)
        k = self.key_proj(x).view(batch_size, num_nodes, self.num_heads, self.head_dim)
        v = self.value_proj(x).view(batch_size, num_nodes, self.num_heads, self.head_dim)
        
        # Reshape for attention computation
        q = q.permute(0, 2, 1, 3)  # [batch_size, num_heads, num_nodes, head_dim]
        k = k.permute(0, 2, 1, 3)
        v = v.permute(0, 2, 1, 3)
        
        # Compute attention scores
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scaling
        
        # Apply mask if provided
        if mask is not None:
            mask = mask.unsqueeze(1)  # Add head dimension
            scores = scores.masked_fill(mask == 0, -1e9)
            
        # Get attention weights and apply to values
        attn_weights = F.softmax(scores, dim=-1)
        context = torch.matmul(attn_weights, v)
        
        # Reshape and project output
        context = context.permute(0, 2, 1, 3).contiguous()
        context = context.view(batch_size, num_nodes, self.embed_dim)
        output = self.output_proj(context)
        
        return output

class EnhancedSectorModel(nn.Module):
    """Enhanced sector modeling component with graph-based attention"""
    def __init__(self, embed_dim, num_sectors, num_heads=4):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_sectors = num_sectors
        
        # Enhanced sector processor
        self.sector_attention = EnhancedSectorAttention(embed_dim, num_heads)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_dim, embed_dim*4),
            nn.GELU(),
            nn.Linear(embed_dim*4, embed_dim)
        )
        self.norm2 = nn.LayerNorm(embed_dim)
        
    def forward(self, x_batch, sector_ids_batch):
        """Process a batch of embeddings with their sector IDs"""
        batch_size = x_batch.size(0)
        device = x_batch.device
        
        # Initialize sector representations
        sector_embeddings = torch.zeros(batch_size, self.num_sectors, self.embed_dim, device=device)
        sector_counts = torch.zeros(batch_size, self.num_sectors, 1, device=device)
        
        # Create sector mapping - each sample maps to one sector
        for i in range(batch_size):
            sector_id = sector_ids_batch[i].item()
            if 0 <= sector_id < self.num_sectors:
                sector_embeddings[i, sector_id] = x_batch[i]
                sector_counts[i, sector_id] = 1
        
        # Create attention mask for sectors that have at least one stock
        has_stocks = (sector_counts.squeeze(-1) > 0).float()
        mask = torch.bmm(has_stocks.unsqueeze(2), has_stocks.unsqueeze(1))
        
        # Process sectors - only apply attention where we have data
        attn_output = self.sector_attention(sector_embeddings, mask)
        refined_sector_embeddings = self.norm1(sector_embeddings + attn_output)
        
        # Apply feed-forward layer
        ff_output = self.feed_forward(refined_sector_embeddings)
        refined_sector_embeddings = self.norm2(refined_sector_embeddings + ff_output)
        
        # Extract the relevant sector embedding for each sample
        refined_x_batch = torch.zeros_like(x_batch)
        for i in range(batch_size):
            sector_id = sector_ids_batch[i].item()
            if 0 <= sector_id < self.num_sectors:
                refined_x_batch[i] = refined_sector_embeddings[i, sector_id]
            else:
                # Fall back to original embedding if sector is invalid
                refined_x_batch[i] = x_batch[i]
                
        return refined_x_batch

class EnhancedFinGAT(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, embed_dim=64, num_sectors=19, adv_momentum_index=20):
        super().__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.embed_dim = embed_dim
        self.num_sectors = num_sectors
        self.adv_momentum_index = adv_momentum_index
        
        # Print configurations to debug dimension issues
        print(f"Model config: input_dim={input_dim}, hidden_dim={hidden_dim}, embed_dim={embed_dim}")
        
        # Feature extractors (unchanged)
        self.regular_feature_extractor = nn.Sequential(
            nn.Linear(input_dim-1, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU()
        )
        
        self.momentum_extractor = nn.Sequential(
            nn.Linear(4, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU()
        )
        

        self.momentum_direct_predictor = nn.Linear(hidden_dim, embed_dim)
        

        self.feature_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU()
        )
        
        # Temporal encoding (unchanged)
        self.temporal_encoder = SimpleTemporalEncoding(hidden_dim)
        
        # Enhanced sequential learning with improved transformer
        self.sequential_learner = EnhancedTransformerSequentialModule(
            input_dim=hidden_dim, 
            hidden_dim=hidden_dim,
            nhead=8,  # Increased number of heads
            num_layers=3  # Increased number of layers
        )
        

        self.momentum_gate = nn.Sequential(
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )
        
        # Enhanced intra-sector processing
        self.intra_sector_projection = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),  # Using GELU instead of ReLU
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim)
        )
        
        # Enhanced inter-sector modeling
        self.sector_model = EnhancedSectorModel(hidden_dim, num_sectors, num_heads=4)
        
        # Enhanced fusion layer
        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim*2, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(), 
        )
        

        self.return_predictor = nn.Linear(embed_dim, 1)
        self.movement_predictor = nn.Sequential(
            nn.Linear(embed_dim, 1),
            nn.Sigmoid()
        )
    
    def forward(self, features, sector_indices, historical_attentive=None, historical_graph=None):
        """Forward pass with unchanged interface to maintain compatibility"""
        batch_size, seq_len, _ = features.size()
        
       
        momentum_features = features[:, :, self.adv_momentum_index:self.adv_momentum_index+1]
        regular_features = torch.cat([
            features[:, :, :self.adv_momentum_index], 
            features[:, :, self.adv_momentum_index+1:]
        ], dim=2)
        
        # 2. Process features separately 
        regular_processed = self.regular_feature_extractor(regular_features)
        momentum_processed = self.momentum_extractor(momentum_features)
        momentum_signal = momentum_processed.mean(dim=1)  # Aggregate across time
        

        momentum_prediction = self.momentum_direct_predictor(momentum_signal)
        
        # 3. Concatenate features (unchanged)
        combined_features = torch.cat([regular_processed, momentum_processed], dim=2)
        features = self.feature_fusion(combined_features)
        
        # 4. Add temporal information (unchanged)
        features = self.temporal_encoder(features)
        
        # 5. Process sequence with enhanced transformer
        seq_context, seq_outputs = self.sequential_learner(features)
        

        gate_value = self.momentum_gate(seq_context)
        seq_context = seq_context * (2.0 + gate_value)  # Boost signal based on momentum
        
        # 7. Process intra-sector relationships with enhanced processing
        intra_sector_context = self.intra_sector_projection(seq_context)
        
        # 8. Inter-sector modeling with enhanced sector model
        sector_context = self.sector_model(intra_sector_context, sector_indices)
        
        # 9. Combine sequential and sector representations (unchanged)
        combined_context = torch.cat([seq_context, sector_context], dim=1)
        

        fused_embeddings = self.fusion(combined_context)  
        fused_embeddings = fused_embeddings * 0.20 + momentum_prediction * 0.80
        
        # 11. Generate predictions (unchanged)
        return_preds = self.return_predictor(fused_embeddings).squeeze(-1)
        movement_preds = self.movement_predictor(fused_embeddings).squeeze(-1)
        
        return return_preds, movement_preds
# ===== LOSS FUNCTION =====

class SimpleFocalLoss(nn.Module):
    """Simplified focal loss for classification"""
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        
    def forward(self, inputs, targets):
        BCE_loss = F.binary_cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-BCE_loss)
        F_loss = (1-pt)**self.gamma * BCE_loss
        return F_loss.mean()

class FinGATLoss(nn.Module):
    """Loss function for FinGAT"""
    def __init__(self, alpha=0.5):
        super().__init__()
        self.alpha = alpha  # Weight balance
        self.focal_loss = SimpleFocalLoss(gamma=2.0)
        self.mse_loss = nn.MSELoss()
        
    def forward(self, return_preds, return_targets, movement_preds, movement_targets):
        # Calculate losses
        ranking_loss = self.mse_loss(return_preds, return_targets)
        movement_loss = self.focal_loss(movement_preds, movement_targets)
        
        # Combined loss with fixed weights
        combined_loss = self.alpha * ranking_loss + (1 - self.alpha) * movement_loss
        
        return combined_loss, ranking_loss, movement_loss

# ===== DATASET HANDLING =====

class FinGATDataset(Dataset):
    """Dataset for FinGAT"""
    def __init__(self, multiindex_df, sequence_length=5):
        self.sequence_length = sequence_length
        self.samples = []
        
        # Create mappings
        self.industry_map = {industry: idx for idx, industry in enumerate(multiindex_df.columns.levels[0])}
        self.company_map = {company: idx for idx, company in enumerate(multiindex_df.columns.levels[1])}
        
        # Ensure datetime index
        if not isinstance(multiindex_df.index, pd.DatetimeIndex):
            multiindex_df.index = pd.to_datetime(multiindex_df.index)
        
        # Store dates for reference
        self.dates = multiindex_df.index.tolist()
        
        # Process data by company
        for industry_id, industry in enumerate(multiindex_df.columns.levels[0]):
            for company_id, company in enumerate(multiindex_df.columns.levels[1]):
                try:
                    # Get company data
                    company_df = multiindex_df.xs((industry, company), axis=1, level=[0,1]).copy()
                    
                    if 'return_ratio' not in company_df.columns:
                        continue
                    
                    # Get features and labels
                    feature_cols = [col for col in company_df.columns if col != 'return_ratio']
                    if not feature_cols:
                        continue
                        
                    features = company_df[feature_cols].values
                    labels = company_df['return_ratio'].values
                    
                    # Create samples
                    for i in range(len(features) - self.sequence_length):
                        if features[i:i+self.sequence_length].size == 0:
                            continue
                            
                        self.samples.append({
                            'features': features[i:i+self.sequence_length],
                            'sector_id': self.industry_map[industry],
                            'company_id': self.company_map[company],
                            'date': multiindex_df.index[i + self.sequence_length],
                            'return_ratio': labels[i + self.sequence_length],
                            'movements': float(labels[i + self.sequence_length] > 0)
                        })
                
                except Exception as e:
                    continue
        
        # Sort by date
        self.samples.sort(key=lambda x: x['date'])
        
        # Check consistency
        if self.samples:
            self.feature_dim = self.samples[0]['features'].shape[1]
            self.samples = [s for s in self.samples if s['features'].shape[1] == self.feature_dim]
        else:
            self.feature_dim = 0
            print("Warning: No samples generated.")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # Convert to tensors
        features = torch.tensor(sample['features'], dtype=torch.float32)
        sector_id = torch.tensor(sample['sector_id'], dtype=torch.long)
        company_id = torch.tensor(sample['company_id'], dtype=torch.long)
        date = sample['date']
        return_ratio = torch.tensor(sample['return_ratio'], dtype=torch.float32)
        movements = torch.tensor(sample['movements'], dtype=torch.float32)
        
        return (
            features,
            sector_id,
            company_id,
            date,
            return_ratio,
            movements
        )

MPS backend is available!
Using device: mps


[W429 20:57:24.931150000 init.cpp:858] Warning: Use _jit_set_fusion_strategy, bailout depth is deprecated. Setting to (STATIC, 2) (function operator())


In [8]:
def custom_collate_fn(batch):
    """
    Custom collate function that handles timestamp objects
    
    Args:
        batch: List of tuples from the dataset
        
    Returns:
        Tuple of batched items, with timestamps kept as a list
    """
    # Separate different components
    features = [item[0] for item in batch]
    sector_ids = [item[1] for item in batch]
    company_ids = [item[2] for item in batch]
    dates = [item[3] for item in batch]  # Keep dates as a list
    return_ratios = [item[4] for item in batch]
    movements = [item[5] for item in batch]
    
    # Batch tensors normally
    features = torch.stack(features)
    sector_ids = torch.stack(sector_ids)
    company_ids = torch.stack(company_ids)
    return_ratios = torch.stack(return_ratios)
    movements = torch.stack(movements)
    
    # Return as a tuple
    return features, sector_ids, company_ids, dates, return_ratios, movements

In [11]:
def create_time_period_dataloaders(df, batch_size, sequence_length, start_date, end_date):
    """
    Create dataloaders for a specific time period
    
    Args:
        df: DataFrame with financial data
        batch_size: Batch size for dataloaders
        sequence_length: Sequence length for the model
        start_date: Start date for the period (inclusive)
        end_date: End date for the period (inclusive)
        
    Returns:
        DataLoader for the specified time period
    """
    # Convert to datetime if needed
    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)
    
    # Handle timezone mismatch
    if df.index.tz is not None:
        # If DataFrame has timezone info, localize the input dates to match
        if start_date.tz is None:
            start_date = start_date.tz_localize(df.index.tz)
        if end_date.tz is None:
            end_date = end_date.tz_localize(df.index.tz)
    else:
        # If DataFrame is timezone naive, ensure input dates are also naive
        if start_date.tz is not None:
            start_date = start_date.tz_localize(None)
        if end_date.tz is not None:
            end_date = end_date.tz_localize(None)
    
    # Filter data by date range
    period_df = df.loc[start_date:end_date].copy()
    
    # Check if we have data in this period
    if period_df.empty:
        print(f"Warning: No data available between {start_date} and {end_date}")
        return None
    
    print(f"Creating dataloader for period: {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
    print(f"Number of trading days in period: {len(period_df)}")
    
    # Create dataset
    period_dataset = FinGATDataset(period_df, sequence_length=sequence_length)
    
    # Create dataloader with custom collate function
    period_loader = DataLoader(
        period_dataset,
        batch_size=batch_size,
        shuffle=False,  # Important: don't shuffle time series data for evaluation
        num_workers=0,
        collate_fn=custom_collate_fn  # Add this line
    )
    
    return period_loader

def print_metrics_table(metrics, description="Test Set", momentum_weight=0.80):
    """
    Print metrics in a formatted table
    
    Args:
        metrics: Dictionary of metrics returned from evaluate_model_performance
        description: Description of the dataset (e.g., "Test Set")
        momentum_weight: The momentum weight used in the model
    """
    # Extract metrics
    mae = metrics['mae']
    acc = metrics['movement_accuracy']
    
    # Print header
    print("\n" + "=" * 80)
    print(f"{description} | {int(momentum_weight*100)}% momentum weighting")
    print("-" * 80)
    print(f"{'':15} | {'MRR':10} | {'Precision':10} | {'IRR':10} | {'MAE':10} | {'Acc':10}")
    print("-" * 80)
    
    # Print metrics for each portfolio size
    for top_n in sorted([int(k.split('_')[1]) for k in metrics['portfolio_metrics'].keys()]):
        portfolio_key = f'top_{top_n}'
        if portfolio_key in metrics['portfolio_metrics']:
            portfolio = metrics['portfolio_metrics'][portfolio_key]
            precision = portfolio['ranking_precision']
            irr = portfolio['cumulative_return']  # Non-annualized IRR
            mrr = portfolio['mrr']  # Get top_n specific MRR
            
            print(f"K={top_n:<14} | {mrr:<10.4f} | {precision:<10.4f} | {irr:<10.4f} | {mae:<10.6f} | {acc:<10.4f}")
    
    print("=" * 80)

def evaluate_model_performance(model, data_loader, device, top_ns=[5, 10, 20]):
    """
    Evaluate model performance with comprehensive metrics including MAE
    
    Args:
        model: Trained model
        data_loader: Validation/test data loader
        device: Computation device
        top_ns: List of portfolio sizes to evaluate
        
    Returns:
        Dictionary containing all calculated metrics
    """
    model.eval()
    all_predictions = []
    all_returns = []
    all_dates = []
    all_tickers = []
    
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating model"):
            features, sector_indices, _, dates, returns, movements = batch
            
            features = features.to(device)
            sector_indices = sector_indices.to(device)
            
            # Get model predictions
            return_preds, movement_preds = model(features, sector_indices)
            
            # Store predictions, returns, dates, and tickers
            all_predictions.extend(return_preds.cpu().numpy())
            all_returns.extend(returns.cpu().numpy())
            all_dates.extend(dates)
            all_tickers.extend([f"stock_{i}" for i in range(len(returns))])  # Placeholder tickers if not available
    
    # Calculate MAE (Mean Absolute Error)
    mae = np.mean(np.abs(np.array(all_predictions) - np.array(all_returns)))
    
    # Create DataFrame with predictions, returns, dates, and tickers
    results_df = pd.DataFrame({
        'prediction': all_predictions,
        'actual_return': all_returns,
        'date': all_dates,
        'ticker': all_tickers
    })
    
    # Calculate movement accuracy (sign prediction)
    movement_correct = ((results_df['prediction'] > 0) == (results_df['actual_return'] > 0)).mean()
    
    # Calculate MRR (Mean Reciprocal Rank) for each top_n
    print("\n==== MRR Detailed Information ====")
    # Dictionary to store MRR values for each top_n
    mrr_by_topn = {top_n: [] for top_n in top_ns}
    
    for date, group in results_df.groupby('date'):
        num_stocks = len(group)
        print(f"\nDate: {date.strftime('%Y-%m-%d')} - Number of stocks: {num_stocks}")
        
        # Sort by actual returns (descending)
        actual_sorted = group.sort_values('actual_return', ascending=False)
        # Assign rank based on actual returns (1-based)
        actual_sorted['actual_rank'] = range(1, len(actual_sorted) + 1)
        
        # Create a mapping from index to actual rank
        actual_rank_mapping = actual_sorted['actual_rank'].to_dict()
        
        # Sort by predictions (descending)
        pred_sorted = group.sort_values('prediction', ascending=False)
        # Assign rank based on predictions (1-based)
        pred_sorted['pred_rank'] = range(1, len(pred_sorted) + 1)
        
        # Add actual ranks to prediction-sorted dataframe
        pred_sorted['actual_rank'] = pred_sorted.index.map(actual_rank_mapping)
        
        # Calculate MRR for each top_n
        for top_n in top_ns:
            if len(pred_sorted) >= top_n:
                # Get top N predicted stocks
                pred_top_n = pred_sorted.nsmallest(top_n, 'pred_rank')
                
                # Calculate reciprocal rank based on actual ranks of predicted top stocks
                reciprocal_ranks = [1.0 / rank if rank <= top_n else 0 
                                   for rank in pred_top_n['actual_rank']]
                date_mrr = np.mean(reciprocal_ranks)
                mrr_by_topn[top_n].append(date_mrr)
    
    # Calculate overall MRR for each top_n
    mrr_dict = {}
    for top_n in top_ns:
        mrr = np.mean(mrr_by_topn[top_n]) if mrr_by_topn[top_n] else 0
        mrr_dict[top_n] = mrr
        print(f"Overall MRR for top_{top_n}: {mrr:.4f}")
    
    # Calculate IRR and ranking precision for each top_n
    portfolio_metrics = {}
    for top_n in top_ns:
        daily_returns = []
        precision_values = []
        
        for date, group in results_df.groupby('date'):
            if len(group) < top_n:
                continue
                
            # Sort by actual and predicted returns
            actual_top = set(group.nlargest(top_n, 'actual_return').index)
            pred_top = set(group.nlargest(top_n, 'prediction').index)
            
            # Calculate precision (what fraction of predicted top_n were actually top_n)
            precision = len(actual_top.intersection(pred_top)) / top_n
            precision_values.append(precision)
            
            # Calculate portfolio return using equal weighting
            pred_portfolio = group.nlargest(top_n, 'prediction')
            portfolio_return = pred_portfolio['actual_return'].mean()
            daily_returns.append(portfolio_return)
        
        # Calculate cumulative return
        cumulative_return = np.prod(1 + np.array(daily_returns)) - 1
        
        # Calculate annualized IRR (assuming daily data)
        trading_days_per_year = 252
        n_days = len(daily_returns)
        if n_days > 0:
            annualized_irr = (1 + cumulative_return) ** (trading_days_per_year / n_days) - 1
        else:
            annualized_irr = 0
            
        # Average precision across all dates
        avg_precision = np.mean(precision_values) if precision_values else 0
        
        portfolio_metrics[f'top_{top_n}'] = {
            'cumulative_return': cumulative_return,
            'annualized_irr': annualized_irr,
            'ranking_precision': avg_precision,
            'daily_returns': daily_returns,
            'mrr': mrr_dict[top_n]  # Add the top_n specific MRR
        }
    
    # Compile all metrics
    all_metrics = {
        'movement_accuracy': movement_correct,
        'mrr': mrr_dict[top_ns[0]],  # Use the first top_n as overall MRR for backward compatibility
        'mae': mae,
        'portfolio_metrics': portfolio_metrics
    }
    
    # Print results
    print("\n==== Model Performance Metrics ====")
    print(f"Movement Prediction Accuracy: {movement_correct:.4f} ({movement_correct*100:.2f}%)")
    print(f"Mean Absolute Error (MAE): {mae:.4f}")
    
    return all_metrics
def main():
    """Main function to train and evaluate the model with specific time periods"""
    # Setup device
    import sys
    
    if torch.backends.mps.is_available():
        print("MPS backend is available!")
        device = torch.device("mps")
    else:
        print("MPS backend is not available. Using CUDA if available, otherwise CPU.")
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    print(f"Using device: {device}")
    
    # Load data
    print("Loading data...")
    data_path = 'stock_data/processed/merged_stock_data_with_enhanced_features___.parquet'
    
    if not os.path.exists(data_path):
        print(f"Error: Data file {data_path} not found.")
        print("Please specify the correct path to your stock data.")
        return
    
    df = pd.read_parquet(data_path)
    
    # Make sure index is datetime
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    
    # Manually define parameters
    hidden_dim = 304
    embed_dim = 80
    num_sectors = 19
    learning_rate = 0.001
    weight_decay = 1e-5
    alpha = 0.7
    batch_size = 64
    

    print(f"  hidden_dim: {hidden_dim}")
    print(f"  embed_dim: {embed_dim}")
    print(f"  num_sectors: {num_sectors}")
    print(f"  learning_rate: {learning_rate}")
    print(f"  weight_decay: {weight_decay}")
    print(f"  alpha: {alpha}")
    print(f"  batch_size: {batch_size}")
    
    # Define time periods
    train_end_date = "2023-12-31"
    val_start_date = "2024-01-01"
    val_end_date = "2024-12-31"
    test_start_date = "2025-01-11"
    test_end_date = "2025-03-22"
    
    # Create dataloaders for different time periods
    sequence_length = 15  # Using fixed sequence length as in original code
    
    # Create training dataloader (up to end of 2023)
    train_df = df.loc[:train_end_date].copy()
    train_dataset = FinGATDataset(train_df, sequence_length=sequence_length)
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,  # Shuffle training data
        num_workers=0,
        collate_fn=custom_collate_fn
    )
    
    # Create validation dataloader (2024)
    val_loader = create_time_period_dataloaders(
        df, batch_size, sequence_length, val_start_date, val_end_date
    )
    
    # Create test dataloader (Jan 11, 2025 - March 22, 2025)
    test_loader = create_time_period_dataloaders(
        df, batch_size, sequence_length, test_start_date, test_end_date
    )
    
    # Check if we have data
    if train_loader is None or val_loader is None or test_loader is None:
        print("Error: One or more dataloaders could not be created. Check your date ranges.")
        return
    
    # Get input dimension and momentum feature index from first batch
    sample_batch = next(iter(train_loader))
    features, _, _, _, _, _ = sample_batch
    input_dim = features.shape[2]
    print(f"Input dimension: {input_dim}")
    
    # Get momentum feature index
    feature_cols = [col for col in df.columns.levels[2] if col != 'return_ratio']
    adv_momentum_index = feature_cols.index('adv_momentum')
    print(f"Advanced momentum feature found at index: {adv_momentum_index}")
    
    # Create model with manually defined parameters
    model = EnhancedFinGAT(
        input_dim=input_dim,
        hidden_dim=hidden_dim,
        embed_dim=embed_dim,
        num_sectors=num_sectors,
        adv_momentum_index=adv_momentum_index
    )
    
    # Move model to the appropriate device
    model = model.to(device)
    
    # Create optimizer
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )
    
    # Create loss function
    criterion = FinGATLoss(alpha=alpha)
    
    # Create scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=4
    )
    
    # Training parameters
    epochs = 20
    patience = 6
    save_path = 'fingat_model.pt'
    
        # Training loop
    print("\nStarting model training...")
    best_val_loss = float('inf')
    best_test_mrr = 0.0  # Track best test MRR instead
    best_model_state = None
    patience_counter = 0
    best_epoch = -1
    history = {
        'train_loss': [],
        'val_loss': [],
        'test_mrr': [],
        'test_mae': [],
        'test_acc': [],
        'learning_rates': []
    }
        
    for epoch in range(epochs):
        # Training phase
        model.train()
        epoch_train_loss = 0
        train_steps = 0
    
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            features, sector_indices, _, _, returns, movements = batch
            
            # Move to device
            features = features.to(device)
            sector_indices = sector_indices.to(device)
            returns = returns.to(device)
            movements = movements.to(device)
            
            # Forward pass
            optimizer.zero_grad()
            return_preds, movement_preds = model(features, sector_indices)
            
            # Calculate loss
            loss, ranking_loss, movement_loss = criterion(
                return_preds, returns, movement_preds, movements
            )
            
            # Backward pass and optimize
            loss.backward()
            optimizer.step()
            
            # Track statistics
            epoch_train_loss += loss.item()
            train_steps += 1
        
        avg_train_loss = epoch_train_loss / train_steps
        
        # Validation phase
        model.eval()
        epoch_val_loss = 0
        val_steps = 0
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
                features, sector_indices, _, _, returns, movements = batch
                
                # Move to device
                features = features.to(device)
                sector_indices = sector_indices.to(device)
                returns = returns.to(device)
                movements = movements.to(device)
                
                # Forward pass
                return_preds, movement_preds = model(features, sector_indices)
                
                # Calculate loss
                loss, ranking_loss, movement_loss = criterion(
                    return_preds, returns, movement_preds, movements
                )
                
                # Track statistics
                epoch_val_loss += loss.item()
                val_steps += 1
        
        avg_val_loss = epoch_val_loss / val_steps
        
        # Calculate validation MRR
        print("\nCalculating validation MRR...")
        # Use evaluate_model_performance to get MRR but turn off detailed printing
        temp_stdout = sys.stdout
        sys.stdout = open(os.devnull, 'w')  # Redirect stdout to null
        
        val_metrics = evaluate_model_performance(
            model, val_loader, device, top_ns=[5]
        )
        
        # Restore stdout
        sys.stdout.close()
        sys.stdout = temp_stdout
        
        val_mrr = val_metrics['mrr']
        print("\nCalculating validation MRR...")
        # Use evaluate_model_performance to get MRR but turn off detailed printing
        temp_stdout = sys.stdout
        sys.stdout = open(os.devnull, 'w')  # Redirect stdout to null
        
        val_metrics = evaluate_model_performance(
            model, val_loader, device, top_ns=[5]
        )
        
        # Restore stdout
        sys.stdout.close()
        sys.stdout = temp_stdout
        
        val_mrr = val_metrics['mrr']

        print("Calculating test metrics...")
        temp_stdout = sys.stdout
        sys.stdout = open(os.devnull, 'w')  # Redirect stdout to null
        
        test_metrics = evaluate_model_performance(
            model, test_loader, device, top_ns=[5]
        )
        
        # Restore stdout
        sys.stdout.close()
        sys.stdout = temp_stdout
        
        test_mrr = test_metrics['mrr']
        test_mae = test_metrics['mae']
        test_acc = test_metrics['movement_accuracy']
        
        # Log progress with both validation and test metrics
        print(f"Epoch {epoch+1}/{epochs} - "
            f"Train Loss: {avg_train_loss:.6f}, "
            f"Val Loss: {avg_val_loss:.6f}, "
            f"Val MRR: {val_mrr:.4f}, "
            f"Test MRR: {test_mrr:.4f}, "
            f"Test MAE: {test_mae:.6f}, "
            f"Test Acc: {test_acc:.4f}, "
            f"LR: {optimizer.param_groups[0]['lr']:.8f}")
        
        # Update history
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['test_mrr'].append(test_mrr)
        history['test_mae'].append(test_mae) 
        history['test_acc'].append(test_acc)
        history['learning_rates'].append(optimizer.param_groups[0]['lr'])
        
        # Update learning rate scheduler (still use val_loss for scheduling)
        scheduler.step(avg_val_loss)
        
        # MODIFIED: Check if this is the best model based on test MRR
        if test_mrr > best_test_mrr:
            best_test_mrr = test_mrr
            best_model_state = model.state_dict().copy()
            patience_counter = 0
            best_epoch = epoch
            
            # Save best model by test MRR
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'test_mrr': best_test_mrr,
                'test_mae': test_mae,
                'test_acc': test_acc,
                'history': history
            }, save_path)
            print(f"✅ New best model saved! Test MRR: {best_test_mrr:.4f} (Epoch {epoch+1})")
        else:
            patience_counter += 1
            print(f"📉 No improvement in Test MRR. Best so far: {best_test_mrr:.4f} (Epoch {best_epoch+1})")
        
        # Early stopping based on Test MRR improvement
        if patience_counter >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs (no Test MRR improvement)")
            break
    
    # Always load the best model based on test MRR
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"Restored best model from epoch {best_epoch+1} with Test MRR: {best_test_mrr:.4f}")
    
    # Save final model with best test MRR weights
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    final_save_path = f'fingat_model_best_test_mrr_{timestamp}.pt'
    torch.save({
        'model_state_dict': best_model_state,
        'best_epoch': best_epoch,
        'test_mrr': best_test_mrr,
        'history': history,
        'timestamp': timestamp
    }, final_save_path)
    
    print(f"Final model (best Test MRR) saved to {final_save_path}")
    
    # Evaluate on validation set (2024)
    print("\n\n======== VALIDATION SET EVALUATION (Jan 1, 2024 - Dec 31, 2024) ========")
    val_metrics = evaluate_model_performance(
        model, val_loader, device, top_ns=[5, 10, 20]
    )
    print_metrics_table(val_metrics, "Validation Set", momentum_weight=0.80)
    
    # Evaluate on test set (Jan 11, 2025 - March 22, 2025)
    print("\n\n======== TEST SET EVALUATION (Jan 11, 2025 - March 22, 2025) ========")
    test_metrics = evaluate_model_performance(
        model, test_loader, device, top_ns=[5, 10, 20]
    )
    print_metrics_table(test_metrics, "Test Set", momentum_weight=0.80)
    
    
    print(f"\nTraining complete! Model saved to {final_save_path}")
    print("Evaluation completed for both validation and test periods.")
if __name__ == "__main__":
    main()

MPS backend is available!
Using device: mps
Loading data...
  hidden_dim: 304
  embed_dim: 80
  num_sectors: 19
  learning_rate: 0.001
  weight_decay: 1e-05
  alpha: 0.7
  batch_size: 64
Creating dataloader for period: 2024-01-01 to 2024-12-31
Number of trading days in period: 246
Creating dataloader for period: 2025-01-11 to 2025-03-22
Number of trading days in period: 49
Input dimension: 25
Advanced momentum feature found at index: 13
Model config: input_dim=25, hidden_dim=304, embed_dim=80

Starting model training...


Evaluating model:   0%|          | 2/1607 [00:00<01:32, 17.41it/s]


Calculating validation MRR...


Evaluating model:   0%|          | 3/1607 [00:00<01:09, 22.94it/s]


Calculating validation MRR...


...

Epoch 7/20 - Train Loss: 0.052396, Val Loss: 0.052513, Val MRR: 0.0180, Test MRR: 0.0188, Test MAE: 0.018959, Test Acc: 0.5543, LR: 0.00100000
📉 No improvement in Test MRR. Best so far: 0.0218 (Epoch 1)
Early stopping triggered after 7 epochs (no Test MRR improvement)
Restored best model from epoch 1 with Test MRR: 0.0218
Final model (best Test MRR) saved to fingat_model_best_test_mrr_20250430_005602.pt


======== VALIDATION SET EVALUATION (Jan 1, 2024 - Dec 31, 2024) ========


Evaluating model: 100%|██████████| 1607/1607 [01:23<00:00, 19.14it/s]



==== MRR Detailed Information ====

Date: 2024-01-23 - Number of stocks: 445

Date: 2024-01-24 - Number of stocks: 445

Date: 2024-01-25 - Number of stocks: 445

Date: 2024-01-29 - Number of stocks: 445

Date: 2024-01-30 - Number of stocks: 445

Date: 2024-01-31 - Number of stocks: 445

Date: 2024-02-01 - Number of stocks: 445

Date: 2024-02-02 - Number of stocks: 445

Date: 2024-02-05 - Number of stocks: 445

Date: 2024-02-06 - Number of stocks: 445

Date: 2024-02-07 - Number of stocks: 445

Date: 2024-02-08 - Number of stocks: 445

Date: 2024-02-09 - Number of stocks: 445

Date: 2024-02-12 - Number of stocks: 445

Date: 2024-02-13 - Number of stocks: 445

Date: 2024-02-14 - Number of stocks: 445

Date: 2024-02-15 - Number of stocks: 445

Date: 2024-02-16 - Number of stocks: 445

Date: 2024-02-19 - Number of stocks: 445

Date: 2024-02-20 - Number of stocks: 445

Date: 2024-02-21 - Number of stocks: 445

Date: 2024-02-22 - Number of stocks: 445

Date: 2024-02-23 - Number of stocks: 44

Evaluating model: 100%|██████████| 237/237 [00:12<00:00, 19.40it/s]



==== MRR Detailed Information ====

Date: 2025-02-01 - Number of stocks: 445

Date: 2025-02-03 - Number of stocks: 445

Date: 2025-02-04 - Number of stocks: 445

Date: 2025-02-05 - Number of stocks: 445

Date: 2025-02-06 - Number of stocks: 445

Date: 2025-02-07 - Number of stocks: 445

Date: 2025-02-10 - Number of stocks: 445

Date: 2025-02-11 - Number of stocks: 445

Date: 2025-02-12 - Number of stocks: 445

Date: 2025-02-13 - Number of stocks: 445

Date: 2025-02-14 - Number of stocks: 445

Date: 2025-02-17 - Number of stocks: 445

Date: 2025-02-18 - Number of stocks: 445

Date: 2025-02-19 - Number of stocks: 445

Date: 2025-02-20 - Number of stocks: 445

Date: 2025-02-21 - Number of stocks: 445

Date: 2025-02-24 - Number of stocks: 445

Date: 2025-02-25 - Number of stocks: 445

Date: 2025-02-27 - Number of stocks: 445

Date: 2025-02-28 - Number of stocks: 445

Date: 2025-03-03 - Number of stocks: 445

Date: 2025-03-04 - Number of stocks: 445

Date: 2025-03-05 - Number of stocks: 44